# Dependências

In [1]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
#!pip install gcloud
#!gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling



c:\Users\ana.sales_republica\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Tratamento

In [2]:
diretorio = 'G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\PNAD\\2025'

In [3]:
os.chdir(diretorio)  

In [6]:
os.listdir(diretorio)

['pnad_indicadores.xlsx']

In [7]:
df = pd.read_excel('pnad_indicadores.xlsx', sheet_name='indicador_pnad_07_v2', nrows=8)
df

,sexo,cor,freq,freq_se,freq_cv,prop,prop_se,prop_cv
0,Homem,Branca,117232.379321,10147.994388,0.086563,0.341450,0.023068,0.067559
1,Homem,Negra,79384.052346,8015.287879,0.100968,0.231213,0.020892,0.090359
2,Homem,Outra,9476.857937,4822.497992,0.508871,0.027602,0.013491,0.488776
3,Mulher,Branca,81820.774382,8590.873359,0.104996,0.238311,0.020494,0.085999
4,Mulher,Negra,52769.723561,5426.328650,0.102830,0.153697,0.015694,0.102111
5,Mulher,Outra,2652.943913,1536.239978,0.579070,0.007727,0.004501,0.582538


In [ ]:
df.columns

In [9]:
df["freq"] = df["freq"].round().astype("Int64") 
df = df.drop(columns=['freq_se', 'freq_cv','prop_se', 'prop_cv'])
df["prop"] = (df["prop"] * 100).round(2)
df['ano'] = 2025
df = df.rename(columns= {'freq':'quantidade_vinculos', 'sexo':'genero','cor':'cor_raca'})   
df = df[['ano','cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df

,ano,cor_raca,genero,quantidade_vinculos,prop
0,2025,Branca,Homem,117232,34.15
1,2025,Negra,Homem,79384,23.12
2,2025,Outra,Homem,9477,2.76
3,2025,Branca,Mulher,81821,23.83
4,2025,Negra,Mulher,52770,15.37
5,2025,Outra,Mulher,2653,0.77


In [10]:
df1 = pd.read_excel('G:\\Drives compartilhados\\República.org\\4. Equipes\\Dados e Comunicação\\DADOS E CONHECIMENTO\\415 - Repositório de Dados\\Repositório Local\\PNAD\\2024\\pnad_indicadores_output_v2.xlsx', sheet_name='Dados7')
df1

,cor,sexo,sum,prop,cv
0,Branca,Homem,106974.783587,58.8,5.539487
1,Branca,Mulher,71083.821453,59.1,8.336432
2,Negra,Homem,72186.303832,39.7,6.251243
3,Negra,Mulher,47657.997900,39.6,9.468591
4,Outra,Homem,2625.781575,1.4,33.100077
5,Outra,Mulher,1504.452713,1.3,57.770890


In [11]:
df1['ano'] = 2024
df1["sum"] = df1["sum"].round().astype("Int64")
df1 = df1.drop(columns=['cv'])
df1 = df1.rename(columns= {'sum':'quantidade_vinculos', 'cor':'cor_raca','sexo':'genero'}) 
df1 = df1[['ano', 'cor_raca', 'genero','quantidade_vinculos', 'prop']]              
df1


,ano,cor_raca,genero,quantidade_vinculos,prop
0,2024,Branca,Homem,106975,58.8
1,2024,Branca,Mulher,71084,59.1
2,2024,Negra,Homem,72186,39.7
3,2024,Negra,Mulher,47658,39.6
4,2024,Outra,Homem,2626,1.4
5,2024,Outra,Mulher,1504,1.3


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ano                  6 non-null      int64  
 1   cor_raca             6 non-null      object 
 2   genero               6 non-null      object 
 3   quantidade_vinculos  6 non-null      Int64  
 4   prop                 6 non-null      float64
dtypes: Int64(1), float64(1), int64(1), object(2)
memory usage: 378.0+ bytes


In [13]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ano                  6 non-null      int64  
 1   cor_raca             6 non-null      object 
 2   genero               6 non-null      object 
 3   quantidade_vinculos  6 non-null      Int64  
 4   prop                 6 non-null      float64
dtypes: Int64(1), float64(1), int64(1), object(2)
memory usage: 378.0+ bytes


In [14]:
df_merged= pd.merge(df, df1, on=['ano', 'cor_raca', 'genero', 'quantidade_vinculos', 'prop'], how='outer')
df_merged

,ano,cor_raca,genero,quantidade_vinculos,prop
0,2024,Branca,Homem,106975,58.80
1,2024,Branca,Mulher,71084,59.10
2,2024,Negra,Homem,72186,39.70
3,2024,Negra,Mulher,47658,39.60
4,2024,Outra,Homem,2626,1.40
5,2024,Outra,Mulher,1504,1.30
6,2025,Branca,Homem,117232,34.15
7,2025,Branca,Mulher,81821,23.83
8,2025,Negra,Homem,79384,23.12
9,2025,Negra,Mulher,52770,15.37


# Upload

In [15]:
client = bigquery.Client(project='repositoriodedadosgpsp')

In [17]:
schema = [bigquery.SchemaField('ano', 'INTEGER', description= 'Ano de referência da observação'),
          bigquery.SchemaField('cor_raca', 'STRING', description= 'Raça/cor autodeclarado ou não'),
          bigquery.SchemaField('genero', 'STRING', description= 'Gênero autodeclarado ou não'),         
          bigquery.SchemaField('quantidade_vinculos', 'INTEGER', description= 'Número total de vinculos observados'),
          bigquery.SchemaField('prop', 'FLOAT', description= 'Proporção de vínculos em relação ao total naquele ano'),
          ]

dataset_ref = client.dataset('cargos_lideranca')

table_ref = dataset_ref.table('PNAD_vinculos_lideranca_genero_cor') 
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df_merged, table_ref, job_config=job_config)
job.result()

LoadJob<project=repositoriodedadosgpsp, location=US, id=7fbe4e0f-6c07-48f1-ab1b-d55d2efa67fe>